# 02 — SmarTRIZ DPO Training

Builds preference pairs from `rejected_dataset.jsonl` + `training_dataset_clean.json`, then fine-tunes the SFT model with DPO.

**Prerequisites:** Notebook 01 must be complete and `checkpoints/sft-7b/merged/` must exist.

In [ ]:
# ─── CONFIG — edit this cell only ────────────────────────
MODEL_SIZE = '7b'
DRIVE_PATH = '/content/drive/MyDrive/smartriz/'
DEEPINFRA_API_KEY = ''   # required only if unmatched cases need teacher generation

SFT_MODEL_DIR = f'{DRIVE_PATH}checkpoints/sft-{MODEL_SIZE}/merged/'
OUTPUT_DIR    = f'{DRIVE_PATH}checkpoints/dpo-{MODEL_SIZE}/'
DPO_DATASET   = f'{DRIVE_PATH}data/dpo_dataset.json'
CLEAN_DATASET = f'{DRIVE_PATH}data/training_dataset_clean.json'
REJECTED_PATH = f'{DRIVE_PATH}data/rejected_dataset.jsonl'
TEACHER_MODEL = 'deepseek-ai/DeepSeek-R1-Distill-Llama-70B'

LORA_R = 16 if MODEL_SIZE == '7b' else 32
LORA_ALPHA = 32 if MODEL_SIZE == '7b' else 64
MAX_LENGTH        = 3072
MAX_PROMPT_LENGTH = 1024
BETA = 0.1
# ──────────────────────────────────────────────────────────

In [ ]:
!pip install -q transformers==4.45.2 peft==0.13.2 trl==0.11.4 bitsandbytes==0.44.1 accelerate==0.34.2 datasets==3.0.1 openai tqdm

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# ── Resume detection: skip DPO pair building if already done
import json
from pathlib import Path

if Path(DPO_DATASET).exists():
    with open(DPO_DATASET) as f:
        dpo_pairs = json.load(f)
    print(f'DPO dataset already built: {len(dpo_pairs)} pairs — skipping build step')
else:
    dpo_pairs = None
    print('DPO dataset not found — will build from rejected + chosen')

checkpoint_dirs = sorted(Path(OUTPUT_DIR).glob('checkpoint-*')) if Path(OUTPUT_DIR).exists() else []
RESUME_FROM = str(checkpoint_dirs[-1]) if checkpoint_dirs else None
print(f'DPO checkpoint: {RESUME_FROM or "none — training from scratch"}')

In [ ]:
if dpo_pairs is None:
    from tqdm.auto import tqdm

    def load_jsonl(path):
        records = []
        with open(path) as f:
            for line in f:
                line = line.strip()
                if line:
                    records.append(json.loads(line))
        return records

    def load_training(path):
        with open(path) as f:
            data = json.load(f)
        return data['cases'] if isinstance(data, dict) else data

    rejected_raw = load_jsonl(REJECTED_PATH)
    chosen_cases = load_training(CLEAN_DATASET)

    # Index chosen cases by first 100 chars of problem for fuzzy matching
    chosen_index = {c.get('problem', '')[:100].strip(): c for c in chosen_cases}

    # Only use judge-stage rejections that have a full problem field
    judge_rejected = [
        r for r in rejected_raw
        if r.get('stage') == 'judge' and r.get('case', {}).get('problem')
    ]
    print(f'Judge-stage rejections with problem field: {len(judge_rejected)}')

    def format_assistant(case):
        cp = case.get('contradiction_pair', {})
        principles = ', '.join(case.get('inventive_principles', []))
        return (
            f'<think>\n{case.get("reasoning_chain", "")}\n</think>\n'
            f'Contradiction:\n\n'
            f'Improving: {cp.get("improving_parameter", "")}\n'
            f'Worsening: {cp.get("worsening_parameter", "")}\n\n'
            f'Inventive Principles: {principles}\n'
            f'Solution:\n{case.get("solution", "")}'
        )

    dpo_pairs = []
    unmatched = []

    for rej in tqdm(judge_rejected, desc='matching chosen/rejected pairs'):
        case = rej['case']
        key = case.get('problem', '')[:100].strip()
        chosen_case = chosen_index.get(key)

        if chosen_case is None:
            unmatched.append(rej)
            continue

        dpo_pairs.append({
            'prompt':   case.get('problem', ''),
            'chosen':   format_assistant(chosen_case),
            'rejected': format_assistant(case),
        })

    print(f'Matched pairs:   {len(dpo_pairs)}')
    print(f'Unmatched cases: {len(unmatched)} (need teacher generation if < 1500 total)')

In [ ]:
# ── Generate chosen responses for unmatched cases via teacher (runs only if needed)
if dpo_pairs is not None and len(dpo_pairs) < 1500 and DEEPINFRA_API_KEY:
    from openai import OpenAI
    from tqdm.auto import tqdm

    client = OpenAI(api_key=DEEPINFRA_API_KEY,
                    base_url='https://api.deepinfra.com/v1/openai')
    TEACHER_SYSTEM = (
        'You are SmarTRIZ. Solve the engineering problem using TRIZ. '
        'Format your response exactly as:\n'
        '<think>\n[step by step reasoning]\n</think>\n'
        'Contradiction:\n\nImproving: [parameter]\nWorsening: [parameter]\n\n'
        'Inventive Principles: [list]\nSolution:\n[detailed solution]'
    )

    need = 1500 - len(dpo_pairs)
    print(f'Generating {min(need, len(unmatched))} chosen responses via teacher...')

    for rej in tqdm(unmatched[:need], desc='teacher generation'):
        case = rej['case']
        try:
            resp = client.chat.completions.create(
                model=TEACHER_MODEL,
                messages=[
                    {'role': 'system', 'content': TEACHER_SYSTEM},
                    {'role': 'user',   'content': case['problem']},
                ],
                temperature=0.3, max_tokens=2048,
            )
            dpo_pairs.append({
                'prompt':   case['problem'],
                'chosen':   resp.choices[0].message.content,
                'rejected': format_assistant(case),
            })
        except Exception as e:
            print(f'Teacher API error: {e}')

    print(f'Total DPO pairs after teacher augmentation: {len(dpo_pairs)}')

# Save DPO dataset
if dpo_pairs is not None:
    with open(DPO_DATASET, 'w') as f:
        json.dump(dpo_pairs, f)
    print(f'Saved {len(dpo_pairs)} preference pairs to {DPO_DATASET}')

In [ ]:
# ── Load SFT merged model + apply fresh LoRA for DPO
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, TaskType

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type='nf4',
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.bfloat16,
)

tokenizer = AutoTokenizer.from_pretrained(SFT_MODEL_DIR, trust_remote_code=True)
model = AutoModelForCausalLM.from_pretrained(
    SFT_MODEL_DIR,
    quantization_config=bnb_config,
    device_map='auto',
    trust_remote_code=True,
)
model.config.use_cache = False

lora_cfg = LoraConfig(
    r=LORA_R,
    lora_alpha=LORA_ALPHA,
    target_modules=['q_proj', 'k_proj', 'v_proj', 'o_proj', 'gate_proj', 'up_proj', 'down_proj'],
    lora_dropout=0.05,
    bias='none',
    task_type=TaskType.CAUSAL_LM,
)
model = get_peft_model(model, lora_cfg)
model.print_trainable_parameters()

In [ ]:
# ── Build HF dataset and run DPOTrainer
from datasets import Dataset as HFDataset
from trl import DPOTrainer, DPOConfig

SYSTEM_PROMPT = (
    'You are SmarTRIZ, an expert engineering innovation assistant. '
    'Solve technical problems using TRIZ methodology. Identify the '
    'technical contradiction, select inventive principles from the '
    'Altshuller matrix, reason step by step, and propose a solution.'
)

def format_dpo_prompt(pair):
    prompt = tokenizer.apply_chat_template(
        [{'role': 'system', 'content': SYSTEM_PROMPT},
         {'role': 'user',   'content': pair['prompt']}],
        tokenize=False, add_generation_prompt=True
    )
    return {'prompt': prompt, 'chosen': pair['chosen'], 'rejected': pair['rejected']}

dpo_hf = HFDataset.from_list([format_dpo_prompt(p) for p in dpo_pairs])
print(f'DPO dataset size: {len(dpo_hf)}')

dpo_config = DPOConfig(
    output_dir=OUTPUT_DIR,
    beta=BETA,
    loss_type='sigmoid',
    max_length=MAX_LENGTH,
    max_prompt_length=MAX_PROMPT_LENGTH,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    learning_rate=5e-5,
    num_train_epochs=1,
    bf16=True,
    logging_steps=10,
    save_steps=100,
    save_total_limit=2,
    report_to='none',
)

dpo_trainer = DPOTrainer(
    model=model,
    ref_model=None,  # implicit reference via PEFT base
    args=dpo_config,
    train_dataset=dpo_hf,
    tokenizer=tokenizer,
)

dpo_trainer.train(resume_from_checkpoint=RESUME_FROM)
print('DPO training complete')

In [ ]:
# ── Merge DPO LoRA into base and save
from pathlib import Path

merged_dpo = OUTPUT_DIR + 'merged/'
if not Path(merged_dpo + 'config.json').exists():
    print('Merging DPO LoRA weights...')
    model = model.merge_and_unload()
    model.save_pretrained(merged_dpo)
    tokenizer.save_pretrained(merged_dpo)
    print(f'DPO merged model saved to {merged_dpo}')
else:
    print(f'DPO merged model already exists at {merged_dpo}')

print('\nReady for Notebook 03 (GGUF conversion + evaluation)')